# Contact Manager Project

This notebook is a small contact manager project.

The main file that we read from is `contacts.csv`.
After that, the notebook can search contacts, filter contacts, find possible duplicates, and export results to new files.

I wrote the code in a very step-by-step way so it is easier to read and follow.


## 1. Import the tools and write the file names

In this project I only need two built-in Python tools:

- `csv` is used to read and write CSV files.
- `SequenceMatcher` is used to compare two names and see if they are almost the same.


In [ ]:
import csv
from difflib import SequenceMatcher

# I keep the file names in variables so they are easy to change later.
contacts_file_name = "contacts.csv"
gmail_file_name = "gmail_contacts.csv"
duplicate_report_file_name = "duplicate_report.txt"


## 2. Load contacts from the CSV file

The CSV file uses semicolons between columns, so I use `delimiter=';'`.

Each row in the CSV file becomes one dictionary. For example, one contact has a name, email, phone, company, and tags.


In [ ]:
def load_contacts(file_name):
    # This list will hold all contacts from the file.
    contacts_list = []

    try:
        with open(file_name, "r", encoding="utf-8", newline="") as file:
            reader = csv.DictReader(file, delimiter=";")

            for row in reader:
                # I make a new dictionary so the important fields are clear.
                contact = {
                    "name": row.get("name", "").strip(),
                    "email": row.get("email", "").strip(),
                    "phone": row.get("phone", "").strip(),
                    "company": row.get("company", "").strip(),
                    "tags": row.get("tags", "").strip(),
                }

                contacts_list.append(contact)

    except FileNotFoundError:
        print("The file was not found:", file_name)

    return contacts_list


In [ ]:
contacts = load_contacts(contacts_file_name)

print("Loaded", len(contacts), "contacts from", contacts_file_name)
print()

if len(contacts) > 0:
    print("First contact in the file:")
    first_contact = contacts[0]

    print("Name:", first_contact["name"])
    print("Email:", first_contact["email"])
    print("Phone:", first_contact["phone"])
    print("Company:", first_contact["company"])
    print("Tags:", first_contact["tags"])
else:
    print("No contacts were loaded.")


## 3. A small display function

I use this function only to print contacts in a clean way.
It helps me avoid writing the same print code many times.


In [ ]:
def show_contacts(contact_list, title):
    print("=" * 70)
    print(title)
    print("=" * 70)

    if len(contact_list) == 0:
        print("No contacts found.")
        return

    counter = 1

    for contact in contact_list:
        print(counter, "-", contact["name"])
        print("   Email:", contact["email"])
        print("   Phone:", contact["phone"])
        print("   Company:", contact["company"])
        print("   Tags:", contact["tags"])
        print()

        counter = counter + 1


## 4. Search functions

These functions search inside the contact list.

I made separate functions instead of one advanced function because this is easier to understand as a beginner.


In [ ]:
def search_by_name(contact_list, search_text):
    found_contacts = []
    search_text = search_text.lower()

    for contact in contact_list:
        contact_name = contact["name"].lower()

        if search_text in contact_name:
            found_contacts.append(contact)

    return found_contacts


def search_by_email(contact_list, search_text):
    found_contacts = []
    search_text = search_text.lower()

    for contact in contact_list:
        contact_email = contact["email"].lower()

        if search_text in contact_email:
            found_contacts.append(contact)

    return found_contacts


def search_by_company(contact_list, search_text):
    found_contacts = []
    search_text = search_text.lower()

    for contact in contact_list:
        contact_company = contact["company"].lower()

        if search_text in contact_company:
            found_contacts.append(contact)

    return found_contacts


def search_by_tag(contact_list, search_text):
    found_contacts = []
    search_text = search_text.lower()

    for contact in contact_list:
        contact_tags = contact["tags"].lower()

        if search_text in contact_tags:
            found_contacts.append(contact)

    return found_contacts


In [ ]:
# Testing the search functions

name_results = search_by_name(contacts, "john")
show_contacts(name_results, "Search results for name: john")

email_results = search_by_email(contacts, "gmail")
print("Number of contacts with gmail in the email:", len(email_results))

company_results = search_by_company(contacts, "Tech")
print("Number of contacts with Tech in the company name:", len(company_results))


## 5. Filter functions

Filtering is close to searching, but here I use it to make useful smaller lists.

Examples:

- contacts that have a specific tag
- contacts that use a specific email domain
- contacts that have a phone number
- contacts that do not have a phone number


In [ ]:
def filter_by_tag(contact_list, wanted_tag):
    filtered_contacts = []
    wanted_tag = wanted_tag.lower()

    for contact in contact_list:
        contact_tags = contact["tags"].lower()

        if wanted_tag in contact_tags:
            filtered_contacts.append(contact)

    return filtered_contacts


def filter_by_email_domain(contact_list, domain):
    filtered_contacts = []
    domain = domain.lower()

    for contact in contact_list:
        contact_email = contact["email"].lower()

        if domain in contact_email:
            filtered_contacts.append(contact)

    return filtered_contacts


def filter_with_phone(contact_list):
    filtered_contacts = []

    for contact in contact_list:
        phone_number = contact["phone"].strip()

        if phone_number != "":
            filtered_contacts.append(contact)

    return filtered_contacts


def filter_without_phone(contact_list):
    filtered_contacts = []

    for contact in contact_list:
        phone_number = contact["phone"].strip()

        if phone_number == "":
            filtered_contacts.append(contact)

    return filtered_contacts


def filter_with_email(contact_list):
    filtered_contacts = []

    for contact in contact_list:
        email_address = contact["email"].strip()

        if email_address != "":
            filtered_contacts.append(contact)

    return filtered_contacts


In [ ]:
# Testing the filter functions

client_contacts = filter_by_tag(contacts, "client")
gmail_contacts = filter_by_email_domain(contacts, "@gmail.com")
contacts_with_phone = filter_with_phone(contacts)
contacts_without_phone = filter_without_phone(contacts)
contacts_with_email = filter_with_email(contacts)

print("Filter results")
print("- Contacts with client tag:", len(client_contacts))
print("- Contacts with Gmail email:", len(gmail_contacts))
print("- Contacts with phone numbers:", len(contacts_with_phone))
print("- Contacts without phone numbers:", len(contacts_without_phone))
print("- Contacts with email addresses:", len(contacts_with_email))


## 6. Find possible duplicate contacts

A duplicate contact means the same person may be saved more than one time.

In this project I check three simple things:

- same email
- same phone number
- very similar names

The name check is not perfect, but it is useful for names like `John Smith` and `Jon Smith`.


In [ ]:
def get_name_similarity(first_name, second_name):
    first_name = first_name.lower().strip()
    second_name = second_name.lower().strip()

    similarity_number = SequenceMatcher(None, first_name, second_name).ratio()

    return similarity_number


def find_duplicates(contact_list):
    duplicate_list = []

    # I use indexes so I can compare each contact with the contacts after it.
    for first_index in range(len(contact_list)):
        first_contact = contact_list[first_index]

        for second_index in range(first_index + 1, len(contact_list)):
            second_contact = contact_list[second_index]
            reasons = []

            first_email = first_contact["email"].lower().strip()
            second_email = second_contact["email"].lower().strip()

            first_phone = first_contact["phone"].strip()
            second_phone = second_contact["phone"].strip()

            if first_email != "" and first_email == second_email:
                reasons.append("Same email")

            if first_phone != "" and first_phone == second_phone:
                reasons.append("Same phone")

            similarity = get_name_similarity(first_contact["name"], second_contact["name"])

            if similarity > 0.85:
                percent = similarity * 100
                reasons.append("Similar names (" + format(percent, ".0f") + "% match)")

            if len(reasons) > 0:
                duplicate_pair = {
                    "contact1": first_contact,
                    "contact2": second_contact,
                    "reasons": reasons,
                }

                duplicate_list.append(duplicate_pair)

    return duplicate_list


In [ ]:
duplicates = find_duplicates(contacts)

print("Found", len(duplicates), "possible duplicate pairs")
print()

pair_number = 1

for duplicate in duplicates:
    contact_a = duplicate["contact1"]
    contact_b = duplicate["contact2"]

    print("Pair", pair_number)
    print("A:", contact_a["name"], "|", contact_a["email"], "|", contact_a["phone"])
    print("B:", contact_b["name"], "|", contact_b["email"], "|", contact_b["phone"])
    print("Reasons:", ", ".join(duplicate["reasons"]))
    print()

    pair_number = pair_number + 1


## 7. Export the results

The last part saves results into files.

I export:

- Gmail contacts to `gmail_contacts.csv`
- duplicate report to `duplicate_report.txt`


In [ ]:
def export_contacts_to_csv(contact_list, file_name):
    if len(contact_list) == 0:
        print("There are no contacts to export.")
        return False

    try:
        with open(file_name, "w", encoding="utf-8", newline="") as file:
            field_names = ["name", "email", "phone", "company", "tags"]
            writer = csv.DictWriter(file, fieldnames=field_names, delimiter=";")

            writer.writeheader()

            for contact in contact_list:
                writer.writerow(contact)

        print("Exported", len(contact_list), "contacts to", file_name)
        return True

    except Exception as error:
        print("Could not export contacts.")
        print("Error:", error)
        return False


def export_duplicates_report(duplicate_list, file_name):
    try:
        with open(file_name, "w", encoding="utf-8") as file:
            file.write("=" * 80 + "\n")
            file.write("DUPLICATE DETECTION REPORT\n")
            file.write("=" * 80 + "\n\n")

            file.write("Found " + str(len(duplicate_list)) + " potential duplicate pairs:\n\n")

            pair_number = 1

            for duplicate in duplicate_list:
                contact_a = duplicate["contact1"]
                contact_b = duplicate["contact2"]
                reasons_text = ", ".join(duplicate["reasons"])

                file.write("Pair " + str(pair_number) + ":\n")
                file.write("  Contact A: " + contact_a["name"] + " | " + contact_a["email"] + " | " + contact_a["phone"] + "\n")
                file.write("  Contact B: " + contact_b["name"] + " | " + contact_b["email"] + " | " + contact_b["phone"] + "\n")
                file.write("  Reasons: " + reasons_text + "\n")
                file.write("-" * 80 + "\n\n")

                pair_number = pair_number + 1

        print("Duplicate report saved to", file_name)
        return True

    except Exception as error:
        print("Could not export duplicate report.")
        print("Error:", error)
        return False


In [ ]:
# Run the exports

gmail_contacts = filter_by_email_domain(contacts, "@gmail.com")

export_contacts_to_csv(gmail_contacts, gmail_file_name)
export_duplicates_report(duplicates, duplicate_report_file_name)


## 8. Final check

This small check shows that the notebook finished the main steps.


In [ ]:
print("Project finished")
print("Total contacts:", len(contacts))
print("Gmail contacts exported:", len(gmail_contacts))
print("Duplicate pairs found:", len(duplicates))
